In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Sujet HMM TP 1 

# Chaînes de Markov cachées (*Hidden Markov Models*, HMM)

 - On suppose que vous avez un ami qui habite dans un autre pays que vous.     
 - Chaque jour, cet ami joue au tennis ou bien bulle dans son canapé, en fonction de la météo.
 - Cependant, vous n'avez pas accès à la météo de ce pays, vous avez accès uniquement à l'information de ce qu'a fait votre ami (tennis ou canapé). 

On parle **d'observations** à propos des actions __tennis/canapé__, et **d'états cachés** pour la météo (3 états cachés ici).


Un **modèle de Markov caché** est un **modèle de séquence de symboles (mots, caractères, tags, phonèmes...) avec de l'information manquante associée à chaque symbole : son état**.

__`Notez que la chaîne de Markov est définie sur les états cachés et pas sur les observations`.__

Il s'agit d'un **modèle joint entre des symboles observables et leurs catégories latentes/cachées/inconnues**. 

Des hypothèses simplificatrices fortes sous-tendent les HMM : 



*   Dépendance de l'état présent à l'état précédent (1er ordre) uniquement
*   Stationnarité : les transitions entre états ne dépendent pas du temps
*   Indépendance statistique des observations entre elles




Dans ce TP, nous reprenons l'exemple du cours, illustré dans cette figure.


<img src="https://www.irit.fr/~Thomas.Pellegrini/ens/M2ML2/cours2/exemple_HMM_meteo_correct.png" alt="chaine Markov" width="500"/>


`Nous noterons les observations`: 

*  *Canapé* $O_0$ 
*  *Tennis* $O_1$ 

`Nous attribuons arbitrairement un indice à chaque état` : 

*   *Pluie* : 0
*   *Nuageux* : 1
*   *Soleil* : 2

On utilise les notations suivantes : 

*   $S_i$, avec $i=0,1,2$ pour désigner l'un de ces trois états.
*   $S^t$, avec $t$ entier positif pour désigner l'état dans lequel on se trouve à l'instant $t$.


Cet exemple est une chaîne de Markov dite d'ordre 1, au sens où la probabilité d'être dans l'état $S^t$ ne dépend que de l'état précédent $S^{t-1}$. Elle serait d'ordre 2, si cette probabilité dépendait de $S^{t-1}$ et $S^{t-2}$.

**Question** : `donner la matrice de transition` $A=[a_{ij}]$ `de cet exemple`. 

Les éléments de $A$ sont les probabilités conditionnelles de passer de l'état $S_j$ à l'état $S_i$ : $a_{ij}=P(S_i|S_j)$.

In [2]:
transition_matrix = torch.tensor([[0.5,0.4,0],[0.3,0.2,0.3],[0.2,0.4,0.7]])
transition_matrix

tensor([[0.5000, 0.4000, 0.0000],
        [0.3000, 0.2000, 0.3000],
        [0.2000, 0.4000, 0.7000]])

En plus de la matrice de transition (entre les états **cachés**), nous introduisons une deuxième matrice, la matrice des probabilités d'obervation que nous notons $B=[B_{ij}]$. Les éléments de $B$ sont les probabilités conditionnelles "d'observer" *tennis* ou *canapé*, en fonction d'un état caché $S_j$ : 

 - $B_{ij}=P(O_i|S_j)$

 - Cette matrice s'appelle la **matrice d'émission**.   
 - Dans la figure, elle encode les probabilités des flêches rouges.

**Question** : `donner la matrice de transition` $B=[a_{ij}]$ `de cet exemple`. 


In [3]:
emission_matrix = torch.tensor([[0.9,0.6,0.2],[0.1,0.4,0.8]])
emission_matrix

tensor([[0.9000, 0.6000, 0.2000],
        [0.1000, 0.4000, 0.8000]])

 - Le HMM de notre exemple est entièrement caractérisé par $\{A, B, \boldsymbol \pi\}$.  
 - Regroupons tout dans cette cellule, en calculant les valeurs de $\boldsymbol \pi$, avec la méthode des valeurs propres par exemple.

In [4]:
# calcul les valeurs propres de la matrice de transition
L, V = torch.linalg.eig(transition_matrix)
pi0 = V[:,0]
#print(f'Valeur propre L : {L}\nVecteurs propre V :\n{V} \n\npi0 = V[:,0]  : {V[:,0]}')
# la distribution stationnaire des  ́etats : le vecteur π
pi0 = pi0/torch.sum(pi0)
#print(f'Distribution stationnaire des états, donne le vecteur π de A : {pi0}')
pi0 = torch.real(pi0)

if torch.allclose(pi0, torch.tensor([0.2182, 0.2727, 0.5091]), atol=1e-04):
  print(f'OK !: {pi0}')
else:
  print('KO !')

OK !: tensor([0.2182, 0.2727, 0.5091])


In [5]:
# ces deux dict seront utiles par la suite
state2ind = {"Pluie": 0, "Nuageux": 1, "Soleil": 2}
ind2state = {0: "Pluie", 1: "Nuageux", 2: "Soleil"}

obs2ind = {"Canapé": 0, "Tennis": 1}
ind2obs = {0: "Canapé", 1: "Tennis"}

## **Question** : générer une séquence aléatoire d'observations de longueur 5

Pour cela, utiliser la fonction ```torch.multinomial()```

https://pytorch.org/docs/stable/generated/torch.multinomial.html

https://fr.wikipedia.org/wiki/Loi_multinomiale
 
 -  En théorie des probabilités, la loi multinomiale (aussi appelée distribution polynomiale1) généralise la loi binomiale.   
 -  La loi binomiale concerne le nombre de succès lors d'une série de n épreuves de Bernoulli indépendantes, comme dans le jeu de pile ou face,    
 -  La loi multinomiale ne se restreint pas aux épreuves comportant deux issues, elle s'applique par exemple au cas de n jets d'un dé à six faces : 
     - l'apparition du chiffre 1 seul peut être modélisée par une loi binomiale 
     - alors que l'ensemble des apparitions des faces 1 à 6 doit être modélisé par une loi multinomiale.

### Stratégie 1 : transition -> émission -> transition -> émission etc

In [21]:
import torch
import torch.nn.functional as F

states = []
# Initialiser l'état
# Génère 1 échantillon aléatoire issu des probabilités fournient par le vecteur stationary_distribution.
# Le vecteur stationnaire fournit les probalités relatives de chaque catégorie.
# torch.multinomial(pi0,1) va tirer un seul échantillon basé sur les probabilités dans pi0.
current_state = torch.multinomial(pi0,1).item()
# Simuler les transitions et émissions
num_steps = 5
# si on veut conserver la suite d'états (optionnel), créer cette liste
states = [ind2state[current_state]] # type: ignore
# liste qui va contenir les observations générées en mots
emissions = []
for _ in range(num_steps):
    # Générer la transition suivante
    next_state = transition_matrix[:,int(current_state)]
    # torch.multinomial(next_state,1) va tirer un seul échantillon basé sur les probabilités de l'état next_state.
    next_state = torch.multinomial(next_state,1).item()
    states.append(ind2state[int(next_state)])
    # Générer l'émission suivante
    given_emission = emission_matrix[:,int(current_state)]
    emission = torch.multinomial(given_emission,1).item()
    emissions.append(ind2obs[int(emission)])
    # Mettre à jour l'état et les émissions
    current_state = next_state

print("États : ", states)
print("Émissions : ", emissions)

États :  ['Soleil', 'Nuageux', 'Pluie', 'Nuageux', 'Soleil', 'Nuageux']
Émissions :  ['Tennis', 'Tennis', 'Canapé', 'Tennis', 'Tennis']


### Stratégie 2 (optionnel) : générer une séquence d'états cachés entière puis générer une séquence d'observations

`Pour générer une séquence d'états cachés puis générer la séquence d'observations dans le cadre d'un HMM suivre les étapes de description détaillée de la stratégie pour illustrer le processus.` 

1°) Étapes pour générer une séquence d'états cachés et d'observations
 - Définir les paramètres du modèle :
   - Matrice de transition : $A$ où $A[i, j]$ représente la probabilité de passer de l'état $i$ à l'état $j$.
   - Matrice d'émission : $B$ où $B[i, k]$ représente la probabilité d'émettre le symbole $k$ étant donné l'état $i$.
   - Distribution initiale : $\large \pi$ qui représente la probabilité de commencer dans chaque état.   

2°) Générer la séquence d'états cachés :
 - Utilisez la matrice de transition $A$ pour générer chaque état suivant.
 - Commencez avec un état initial choisi selon la distribution  $\large \pi$ .   
 
3°) Générer la séquence d'observations :
 - Utilisez la matrice d'émission $B$ pour générer chaque observation étant donné l'état courant.

In [23]:
# Générer la séquence d'états cachés
num_steps = 5
transitions = []

# Initialiser l'état
# Génère 1 échantillon aléatoire issu des probabilités fournient par le vecteur stationary_distribution.
# Le vecteur stationnaire fournit les probalités relatives de chaque catégorie.
# torch.multinomial(pi0,1) va tirer un seul échantillon basé sur les probabilités dans pi0.

current_state = torch.multinomial(pi0,1).item()
states.append(ind2state[int(current_state)])

for _ in range(num_steps):
    next_state = transition_matrix[:,int(current_state)]
    next_state = torch.multinomial(next_state, 1).item()
    transitions.append(ind2state[int(next_state)])
    current_state = next_state

# Générer la séquence d'observations
observations = []
for state in transitions:
    emission_given = emission_matrix[:, state2ind[state]]
    observation = torch.multinomial(emission_given, 1).item()
    observations.append(ind2obs[int(observation)])

print("Séquence d'états cachés : ", transitions)
print("Séquence d'observations : ", observations)    

Séquence d'états cachés :  ['Soleil', 'Soleil', 'Soleil', 'Soleil', 'Soleil']
Séquence d'observations :  ['Canapé', 'Tennis', 'Tennis', 'Canapé', 'Tennis']


### `Question :` 
`Comment calculer la probabilité d'une séquence d'observations données, lorsque la séquence d'états cachés associée est connue`

Cela signifie qu'au lieu d'évaluer seulement la vraisemblance des observations sachant un chemin d’états fixé, on veut évaluer la probabilité de l’événement complet “le chemin d’états ET les observations”, c’est‑à‑dire la quantité conjointe, et pas seulement la conditionnelle. 

__`Probabilté conditionnelle : `__ $\color{red}{P(x^0,...,x^T|y^0,...,y^T)= \prod_{t=0}^T P(x^t∣y^t)}$

La probabilité conditionnelle ignore la probabilité d’avoir ce chemin $y^{0:T}$ en premier lieu, et ne mesure que la compatibilité des observations avec ce chemin s’il est imposé. On choisi une séquence d'observations x (0,1,1) et on effectue le produit sachant y.

__`Probabilité conjointe : `__ $\color{blue}{P(x^0,...,x^T|y^0,...,y^T)= \alpha (y^0) P(x^0∣y^0) \prod_{t=1}^T P(y^t∣y^{t−1})P(x^t∣y^t)}$

La probabilité conjointe combine la probabilité d’entrer dans l’état initial, de suivre les transitions du chemin, et d’émettre les observations.  

où : 

  . $\color{blue}{\alpha(y^0)}$ : probabilité initiale d’être dans l'état $y^0$.   
  . $\color{blue}{P(y^t∣y^{t-1})}$ : probabilité de transition d'un état à un autre.   
  . $\color{blue}{P(x^t∣y^t)}$ : probabilité d'émission de l'obvservation sachant l'état caché. 

Lorsque la séquence d’états cachés est connue, il suffit d’appliquer la formule directe du produit des probabilités.

$\color{blue}{P(x^0,x^1,x^2|y^0,y^1,y^2)=\alpha(y^0)P(x^0∣y^0)*P(y^1∣y^0)P(x^1∣y^1)*P(y^2∣y^1)P(x^2∣y^2)}$

<img src="hmm.png" alt="forward" width="600"/>

## Tutoriel HMM : le tutoriel "Rabiner" (1989)


*A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition* par Lawrence R. Rabiner, publié en 1989 dans Proc. of the IEEE.

Une version est disponible ici : https://www.cs.ubc.ca/~murphyk/Bayes/rabiner.pdf

Rabiner y décrit trois problèmes  (section III) fondamentaux liés aux HMM : 


1.   *<span style="color:blue">Évaluation* : Étant donnée une séquence d'observations $O = O^0, ..., O^{T-1}$, et un modèle HMM $\lambda = \{\boldsymbol \pi, A, B\}$,   
comment peut-on calculer efficacement la probabilité $P(O|\lambda)$ de la séquence $O$, étant donné le modèle ?

2.   *<span style="color:blue">Inférence ou décodage* : Étant donnée une séquence d'observations $O = O^0, ..., O^{T-1}$, et un modèle HMM $\lambda = \{\boldsymbol \pi, A, B\}$, comment retrouver la séquence d'états $S = S^0, ..., S^{T-1}$ optimale pour expliquer la séquence d'observations $O$ ?

3.   *<span style="color:blue">Apprentissage section IV* : comment ajuster les paramètres du modèle $\lambda = \{\boldsymbol \pi, A, B\}$ pour maximiser la vraisemblance (likelihood) de données d'entraînement,    
c'est-à-dire maximiser $P(O|\lambda)$.
 
Ces trois problèmes font appel à des algorithmes très astucieux : 

1.   *<span style="color:green">Évaluation* : Algorithme *forward* (il existe aussi la version *backward*, utile pour le problème 3)
2.   *<span style="color:green">Inférence ou décodage</span>* : algorithme de *Viterbi* 
3.   *<span style="color:green">Apprentissage</span>**. Deux situations possibles :

      a.  *<span style="color:green">Apprentissage supervisé</span>* : on a les séquences $O$ et $S$ correspondantes. Alors on utilise l'apprentissage par maximum de vraisemblance, qui donne des formules analytiques pour calculer $\boldsymbol \pi, A, B$ à partir des fréquences d'occurrences des observations et des états.
      
      b.   *<span style="color:green">Apprentissage non-supervisé</span>* : on a des séquences $O$ mais pas $S$. Algorithme de type *Expectation-Maximization* (EM) appelé la méthode *Baum-Welch* dans le tutoriel.


Les algorithmes *forward* et *Viterbi* sont des algorithmes de programmation dynamique, tout comme les algorithmes de distance de Levenshtein minimale, de recherche du plus court chemin de Dijkstra, etc. 


Dans ce TP, nous nous intéresserons seulement aux deux premiers problèmes de Rabiner, et coder les deux algorithmes *forward* et *Viterbi*. S'il reste du temps, il y a aussi en exercice le décodage *posterior*.



## Problème 1 : **Évaluation**, coder l'algorithme *forward*

Étant donnée une séquence d'observations $O = O^0, ..., O^{T-1}$, et un modèle HMM $\lambda = \{\boldsymbol \pi, A, B\}$, comment peut-on calculer efficacement la probabilité $P(O|\lambda)$ de la séquence $O$, appelée vraisemblance ou *likelihood*, étant donné le modèle ?

L’algorithme forward est une procédure pour calculer efficacement la probabilité totale d’observer une séquence donnée dans un HMM, en marginalisant sur tous les chemins d’états cachés possibles.

__`Principe`__:  On définit pour chaque étape $t$ et chaque état caché $y$, la variable "forward". C’est la probabilité d’avoir généré les observations jusqu’à l’instant $t$ et de se trouver dans l’état caché 
$i$ à cet instant $t$.     

$\color{red}{\alpha^t(i)=P(x_0,…,x_t,y_t=i)}$.   

__`Initialisation`__: Pour chaque état $i$

$ \color{green}{\alpha_0(i) = \Pi(i)*P(x_0|y_0=i)} $  où $\color{green}{\Pi(i)}$ est la probabilité initiale d’être dans l’état i.

__`Récurrence`__: Pour $t \geqslant 1$ et pour chaque état caché $i$.  

$\color{blue}{\alpha^t(i)= P(x^t|y=i) \sum_{j=1}^N \alpha^{t-1}(j) * P(y^t=i|y^{t-1}=j)}$

 . $\color{blue}{P(x^t∣y^t=i)}$ est la proba d’émission du symbole observé $x^t$ dans l’état i,     
 . $\color{blue}{P(y^t=i∣y^{t−1}=j)}$ est la proba de transition de l’état $j$ à l’état $i$.

On parcourt tous les états précédents $j$, on multiplie leur probabilité d’être atteints par la probabilité de transition vers $i$, puis on multiplie par la probabilité d’émettre l’observation $x^t$ depuis $i$.

__`Terminaison`__ : La probabilité globale d’observer la séquence $(x^0,…,x^T)$ est :

$\color{orange}{P(x^0,…,x^T)=\sum_y \alpha^T(y)}$

On sommme sur tous les états cachés à la dernière étape.



### Approche 1 : force brute (pas de code à écrire)


Considérons la séquence *Canapé -> Tennis*, lister tous les cas à envisager et les probabilités associées.

`Force brute` :
 - considérer toutes les séquences d'états valides, qui pourraient générer $O$ et sommer les probabilités correspondantes. 
 - Il y a 3x3 cas soit 9 cas, on note `U` pour Pluie, `N` pour Nuageux et `S` pour Soleil, on doit calculer :



$
\begin{aligned}
P(01,UU)&=&\boldsymbol \Pi(U)*P(0|U)*P(U|U)*P(1|U)\\
P(01,UN)&=&\boldsymbol \Pi(U)*P(0|U)*P(N|U)*P(1|N)\\
P(01,US)&=&\boldsymbol \Pi(U)*P(0|U)*P(S|U)*P(1|S)\\
P(01,NU)&=&\boldsymbol \Pi(N)*P(0|N)*P(U|N)*P(1|U)\\
P(01,NN)&=&\boldsymbol \Pi(N)*P(0|N)*P(N|N)*P(1|N)\\
P(01,NS)&=&\boldsymbol \Pi(N)*P(0|N)*P(S|N)*P(1|S)\\
P(01,SU)&=&\boldsymbol \Pi(S)*P(0|S)*P(U|S)*P(1|U)\\
P(01,SN)&=&\boldsymbol \Pi(S)*P(0|S)*P(N|S)*P(1|N)\\
P(01,SS)&=&\boldsymbol \Pi(S)*P(0|S)*P(S|S)*P(1|S)\\
\end{aligned}
$

On note des redondances, nous posons donc :    
   
$\alpha_U = \boldsymbol \pi(U) * P(0|U)$   
$\alpha_N = \boldsymbol \pi(N) * P(0|N)$      
$\alpha_S = \boldsymbol \pi(S) * P(0|S)$    

In [9]:
alpha_U = pi0[0] * emission_matrix[0,0]  
alpha_N = pi0[1] * emission_matrix[0,1]     
alpha_S = pi0[2] * emission_matrix[0,2] 

print(f'alpha_U : {alpha_U} alpha_N : {alpha_N} alpha_S :{alpha_S}')

alpha_U : 0.19636359810829163 alpha_N : 0.16363637149333954 alpha_S :0.10181819647550583


Ce qui nous donne :    

$
\begin{aligned}
P(01, UU) &=& \alpha_U & P(U|U) * P(1|U)\\
P(01, NU) &=& \alpha_N & P(U|N) * P(1|U)\\
P(01, SU) &=& \alpha_S & P(U|S) * P(1|U)\\
P(01, UN) &=& \alpha_U & P(N|U) * P(1|N)\\
P(01, NN) &=& \alpha_N & P(N|N) * P(1|N)\\
P(01, SN) &=& \alpha_S & P(N|S) * P(1|N)\\
P(01, US) &=& \alpha_U & P(S|U) * P(1|S)\\
P(01, NS) &=& \alpha_N & P(S|N) * P(1|S)\\
P(01, SS) &=& \alpha_S & P(S|S) * P(1|S)\\
\end{aligned}
$

Faisons les calculs

In [32]:
P_01_UU = alpha_U * 0.5 * 0.1
P_01_NU = alpha_N * 0.4 * 0.1
P_01_SU = alpha_S * 0.0 * 0.1
P_01_UN = alpha_U * 0.3 * 0.4
P_01_NN = alpha_N * 0.2 * 0.4
P_01_SN = alpha_S * 0.3 * 0.4
P_01_US = alpha_U * 0.2 * 0.8
P_01_NS = alpha_N * 0.4 * 0.8
P_01_SS = alpha_S * 0.7 * 0.8

somme = P_01_UU + P_01_NU + P_01_SU + P_01_UN + P_01_NN + P_01_SN + P_01_US + P_01_NS + P_01_SS
print(f'somme : {somme}')

somme : 0.20603635907173157


Séquence d'observations :  ['Canapé', 'Tennis', 'Canapé'], 

`Etape 1 encoderla séquence`

 - donc $ x = (0,1,0)$

`Étape 2 énumérer tous les chemins d’états cachés`

Pour 3 observations avec 3 états possibles (U, N, S), il y a $3^3$ = 27 combinaisons possibles de chemins d’états !

Chaque chemin :    

  - $y=(y^0,y^1,y^2)$ est une séquence de U/N/S à chaque étape.

`Étape 3 calculer chaque terme de la conjointe`

Pour chaque chemin d’états : 
 - $P(x,y)=\Pi(y^0)⋅P(x^0∣y^0)⋅P(y^1∣y^0)⋅P(x^1∣y^1)⋅P(y^2∣y^1)⋅P(x^2∣y^2)$

`Étape 4 tout sommer`
La probabilité totale d’observer cette séquence est la somme sur tous les chemins :

 - $P(x)= \sum_{y^0,y^1,y^2} P(x,y)$

C’est ce que fait l’algorithme forward : il évite ce calcul exhaustif en le traitant de façon dynamique.

Bilan


Pour une séquence de longueur $T$ et $N$ états cachés, le nombre de multiplications à faire pour calculer cette probabilité est de l'ordre de $2TN^T$.

Autrement dit, la complexité est $O(N^T)$, soit exponentielle par rapport au nombre d'états cachés.

Il faut donc faire autrement...

### Approche 2 : l'algorithme forward (à vous de jouer)

Si l'on énumère les séquences d'états possibles pour générer une séquence d'observation, on s'aperçoit qu'il y a beaucoup de calculs en commun, et donc redondants.

L'algorithme *forward* est un algorithme de programmation dynamique.

Il utilise une variable intermédiaire appelée $\alpha$ :

$\alpha^t(S_i)$ est la probabilité jointe de la séquence partielle d'observations $O^0, \ldots, O^t$ tout en étant dans l'état $S_i$ à l'instant $t$ : 

$\alpha^t(S_i) = P(O^0, \ldots, O^t, S^t=S_i)$

L'algorithme *forward* est le suivant : 



*   Initialisation

   *   Pour tous les états $i=0\ldots N-1$, poser :  $\alpha^0(S_i) = \boldsymbol \pi_i P(O^0|S_i)$

À noter que $P(O^0|S_i)$ est un élément de la matrice d'émission $B$. 

*   Récurrence/induction

    *   Pour les itérations suivantes, d'indices $t=1\ldots T-1 $, et pour les $N$ états $i=0\ldots N-1$, calculer :

$$\alpha^t(S_i) = \sum_{j=0}^{N-1} \alpha^{t-1}(S_j)\,P(S_i|S_j)\,P(O^t|S_i) = P(O^t|S_i) \sum_{j=0}^{N-1} \alpha^{t-1}(S_j)\,P(S_i|S_j)$$

*   Terminaison

    *   On obtient finalement la probabilité de la séquence complète : 

$$P(O) = \sum_{j=0}^{N-1} \alpha^{T-1}(S_j)$$





<img src="https://www.irit.fr/~Thomas.Pellegrini/ens/M2ML2/cours2/forward.png" alt="forward" width="200"/>





#### Coder une première version "naïve", avec des boucles *for* imbriquées pour l'étape d'induction.

Votre fonction prend en entrée une séquence d'observations et les paramètres du modèle, et retourne la probabilité de la séquence.

In [33]:
# Calculer la probabilité de la séquence d'observations
def forward_naif(observations, transition_matrix, emission_matrix, pi0):
    num_states = len(pi0)
    num_observations = len(observations)
    alpha = torch.zeros(num_observations, num_states)
    # Initialisation
    print(f'pi0 : {pi0}')
    for x in range(num_states):
        alpha[0,x]= pi0[x].item() * emission_matrix[obs2ind[observations[0]],x].item() 
    # Itération
    for t in range(1,num_observations):
        for i in range(num_states):
            RS = 0
            for j in range(num_states):
                RS += alpha[t-1,j] * transition_matrix[i,j] * emission_matrix[obs2ind[observations[t]],i]
            alpha[t, i] = RS
    # Probabilité de la séquence d'observation
    probability = alpha[-1].sum()
    
    return probability

Tester votre fonction :

In [35]:
seq_obs = [0,1]
seq_obs_mots = [ind2obs[i] for i in seq_obs]

p = forward_naif(seq_obs_mots, transition_matrix, emission_matrix, pi0)
print(" -> ".join(seq_obs_mots))
print("Probabilité : {:.3f}".format(p))

if torch.isclose(p, torch.tensor(0.206), atol=1e-03):
  print('OK !')
else:
  print('KO !')

pi0 : tensor([0.2182, 0.2727, 0.5091])
Canapé -> Tennis
Probabilité : 0.206
OK !


#### Amélioration

Dans l'étape d'induction, ne garder que la boucle sur les éléments de la séquence et remplacer les autres boucles imbriquées par un produit matriciel.

In [36]:
def forward(observations, transition_matrix, emission_matrix, pi0):
    # Nombre d'états
    num_states = transition_matrix.size(0)
    # Nombre d'observations
    num_observations = len(observations)
    alpha = torch.zeros(num_observations,num_states)  
    alpha[0] = pi0 * emission_matrix[observations[0]]
    # Itérer à travers la séquence
    for t in range(1, num_observations):
        alpha[t] = (transition_matrix @ alpha[t-1]) * emission_matrix[observations[t]]
    return torch.sum(alpha[-1])

Tester votre fonction

In [37]:
#observations = [0,1]
observations = [0,1]
seq_obs_mots = [ind2obs[i] for i in observations]

p = forward(observations, transition_matrix, emission_matrix, pi0)
print(" -> ".join(seq_obs_mots))
print("Probabilité : {:.3f}".format(p))
if torch.isclose(p, torch.tensor(0.206), atol=1e-03):
  print('OK !')
else:
  print('KO !')

Canapé -> Tennis
Probabilité : 0.206
OK !


Tester votre fonction sur une séquence plus longue

In [38]:
def forward(observations, transition_matrix, emission_matrix, pi0):
    # Nombre d'états
    num_states = transition_matrix.size(0)
    # Nombre d'observations
    num_observations = len(observations)
    alpha = torch.zeros(num_observations,num_states)  
    alpha[0] = pi0 * emission_matrix[observations[0]]
    # Itérer à travers la séquence
    for t in range(1, num_observations):
        alpha[t] = (transition_matrix @ alpha[t-1]) * emission_matrix[observations[t]]
    return torch.sum(alpha[-1])

In [39]:
#observations = [0,1]
observations = [0,1,1,1,0,1]
seq_obs_mots = [ind2obs[i] for i in observations]

p = forward(observations, transition_matrix, emission_matrix, pi0)
print(" -> ".join(seq_obs_mots))
print("Probabilité : {:.3f}".format(p))

if torch.isclose(p, torch.tensor(0.014), atol=1e-03):
  print('OK !')
else:
  print('KO !')

Canapé -> Tennis -> Tennis -> Tennis -> Canapé -> Tennis
Probabilité : 0.014
OK !


**Question** : pourquoi est-ce que la probabilité de cette séquence est plus petite que celle de la séquence précédente ?

La complexité de cet algorithme est $O(N^2T)$, bien mieux que $O(N^T)$ !!

## Problème 2 : **Inférence ou décodage**
| Terme    | But               | Questions posées  | Algorithme typique|
|----------|-------------------|-------------------|---------------|
| Inférence|Probabilités sur les états cachés|“À un instant donné, quelles sont les chances d’être dans chaque état ?”| Forward/Backward|
| Décodage | Recherche du chemin d’états le plus probable|“Quel est le chemin caché le plus probable ?”| Viterbi (dynamique)|


__`Inférence`__

__Inférer, dans un HMM, signifie__ : Calculer la distribution (ou probabilité) sur les états cachés, compte tenu d’une séquence d’observations.

On peut vouloir : 
 - La probabilité de chaque état à chaque instant, inférence marginale, distribution sur les états, via Forward-Backward,
 - La distribution sur les chemins entiers,
 - Ou encore la vraisemblance globale d'une séquence observée.

__Exemple d’inférence :__ Quelle est la probabilité d’être dans l’état “Nuageux” à t=3, $P(y^3 = Nuageux∣x^{0:T})$ sachant toute la séquence d’observations.

__`Décodage`__

__Décodage signifie :__ 

 - Retrouver le meilleur chemin d’états cachés expliquant la séquence d’observations.
 - Ça répond à la question : “Compte tenu de ce que j’ai observé, quelle série d’états est la plus probable globalement ?”

__Exemple de décodage :__ Pour la séquence “Canapé, Tennis, Canapé”, le chemin d’états le plus probable serait : (Soleil, Nuageux, Soleil) $argmax_y^{0:T} P(y^{0:T}∣x^{0:T})$

### Exemple tennis ou canapé?

Nous souhaitons écrire un algorithme qui remplisse le tableau suivant avec les scores de chaque case, ainsi que, dans un deux temps, l'état d'où l'on vient à l'instant d'avant, pour obtenir la séquence d'états optimale.

En effet, ce qui nous intéresse dans le décodage, c'est d'obtenir la séquence de prédictions des états cachés.

<img src="https://www.irit.fr/~Thomas.Pellegrini/ens/M2ML2/cours2/viterbi_meteo.png" alt="viterbi" width="400"/>

#### Exemple de cas d'usage : en traitement de données textuelles (*NLP* pour *Natural Language Processing*), trouver les tags Part-Of-Speech des mots d'une phrase (*POS tagging*) : 

<img src="https://www.irit.fr/~Thomas.Pellegrini/ens/M2ML2/cours2/problem2_exemple_POS_tagging.png" alt="viterbi" width="400"/>

4000 séquences d'états possibles sur ce petit exemple !

(Image de Noah Smith)

### Formulation

Étant donnée une séquence d'observations $O = O^0, ..., O^{T-1}$, et un modèle HMM $\lambda = \{\boldsymbol \pi, A, B\}$, comment retrouver la séquence d'états $S = S^0, ..., S^{T-1}$ optimale pour expliquer la séquence d'observations $O$ ?


Nous pourrions vouloir simplement choisir l'état le plus vraisemblable à chaque temps $t$, indépendamment des états précédents ou suivants : selon $P(S^t=S_i|O)$. Ce serait très efficace, mais cela peut donner des séquences non-valides, si par exemple il y a des probabilités de transition nulles dans le modèle.

L'algorithme de référence est l'algorithme Viterbi, qui est un algorithme de programmation dynamique.



### Viterbi (scores seulement)

Nous allons calculer à chaque temps $t$, le meilleur score (plus grande vraisemblance) d'un unique meilleur chemin, qui rend compte des $t$ premières observations, et qui aboutit à l'état $S_i$ au temps $t$.    

Dans le tutoriel Rabiner, ce score est noté avec la lettre delta :  $\color{red}{\delta^t(S_i) = \max_{S^0, S^1, \ldots, S^{t-1}} P(S^0, O^0, S^1, O^1, \ldots, S^t=S_i, O^t)}$

*   __Initialisation__

      *   Pour tous les états $\color{red}{i=0\ldots N-1}$, poser :    
            *  $\color{red}{\delta^0(S_i) = \boldsymbol \pi_i P(O^0|S_i)}$
            *  À noter que $\color{red}{P(O^0|S_i)}$ est un élément de la matrice d'émission $\color{red}{B}$. 

*   __Récurrence/induction__

      *   Pour les itérations suivantes, d'indices $\color{red}{t=1\ldots T-1} $, et pour les $\color{red}{N}$ états $\color{red}{i=0\ldots N-1}$, calculer :

           *    $\color{red}{\delta^t(S_i) = \max_{j=0\ldots N-1} \delta^{t-1}(S_j)\,P(S_i|S_j)\,P(O^t|S_i) = P(O^t|S_i) \max_{j=0\ldots N-1} \delta^{t-1}(S_j)\,P(S_i|S_j }$

*   __Terminaison__

    *   Obtention finale du score de la séquence la plus vraisemblable des états : 

           *    $\color{red}{\text{score}^{T-1} = P(S^*|O) = \max_{j=0\ldots N-1} \delta^{T-1}(S_j) }$



#### Exemple POS tagging

<img src="https://www.irit.fr/~Thomas.Pellegrini/ens/M2ML2/cours2/viterbi_example_POS_scores.png" alt="viterbi POs example" width="400"/>


(Image de Noah Smith)

#### Code **Exercice** 

Votre fonction prend en entrée une séquence d'observations et les paramètres du modèle, et retourne le score de la séquence.

In [68]:
def viterbi_score(observations, transition_matrix, emission_matrix, pi0):
    # observations : sequence d'observations, liste d'entiers
    # transition_matrix : matrice de transition N x N où N nb d'états cahcés
    # emission_matrix : matrice d'émission O x N où O nb d'observations possibles différentes
    # pi0: vecteur distribution stationnaire des N états cachés
    
    num_states = transition_matrix.size(0)
    num_observations = len(observations)   
    # Initialiser 
    alpha = torch.zeros((num_observations, num_states)) 
    #print(f'alpha :\n{alpha},\nnum_observations : {num_observations}\nnum_states : {num_states}\n')
    # Initialisation
    alpha[0] = pi0 * emission_matrix[observations[0]]
    #print(f'\nalpha[0] : {alpha[0]}\n')
    # induction
    for t in range(1,num_observations):
        for i in range(num_states):
            prob = 0
            max_prob = 0
            for j in range(num_states):
                #print(f'j : {j} - alph[{t-1},{j}] * transition_matrix[{i}|{j}] * emission_matrix[{t}|{i}]')       
                prob = alpha[t-1,j] * transition_matrix[i,j] * emission_matrix[observations[t],i]
                #print(f' {prob} = {alpha[t-1,j]} * {transition_matrix[i, j]} * {emission_matrix[observations[t],i]}')   
                if prob > max_prob:
                    max_prob = prob         
            alpha[t, i] = max_prob   
    #res = alpha[t-1:torch.argmax(alpha[t-1])] 
    res = alpha[-1,torch.argmax(alpha[num_observations-1])]     
    # Trouver les indices de la plus haute valeur
    return res


In [69]:
transition_matrix = torch.tensor([[0.5,0.4,0],[0.3,0.2,0.3],[0.2,0.4,0.7]])
emission_matrix = torch.tensor([[0.9,0.6,0.2],[0.1,0.4,0.8]])
observations = [0,1]
seq_obs_mots = [ind2obs[i] for i in observations]

score = viterbi_score(observations, transition_matrix, emission_matrix, pi0)
print(" -> ".join(seq_obs_mots))
print("Score Viterbi : {:.3f}".format(score))

if torch.isclose(score, torch.tensor(0.057), atol=1e-03):
  print('OK !')
else:
  print('KO !')

Canapé -> Tennis
Score Viterbi : 0.057
OK !


#### **Question : est-ce qu'on n'a pas fait la même chose déjà avec l'algorithme *forward* ?**

`Il n’exite pas de Viterbi backward, mais il n’y a pas de raison à cela`

<img src="https://www.irit.fr/~Thomas.Pellegrini/ens/M2ML2/cours2/diff_forward_viterbi.png" alt="chaine Markov" width="300"/>


### Viterbi (scores et séquence d'états)
Viterbi nous donne le score du meilleur chemin, mais comment avoir les prédictions (la séquence d'états cachés) qui correspondent ?

Lors du calcul des scores par induction, à chaque temps $t$, il faut conserver l'identité de l'état dont le score est maximal. 

Ces états seront notés $\text{bp}$ pour *backpointers* en anglais.

Voici l'algo complété.

*   Initialisation

   *   Pour tous les états $i=0\ldots N-1$, poser :
  $\delta^0(S_i) = \boldsymbol \pi_i P(O^0|S_i)$ 
        *   $\delta^0(S_i) $ probabilité Viterbi au temps 0 d’être dans l’état $S_i$.   
        *   $\pi_i$ probabilité initiale
        *   $P(O^0|S_i)$ probabilité d'émission de $S_i$ sachant l'état 0 (élément de la matrice d'émission $B$). 

*   Récurrence/induction

    *   Pour les itérations suivantes, d'indices $t=1\ldots T-1 $, et pour les $N$ états $i=0\ldots N-1$, calculer :

        *  $\delta^t(S_i) = \max_{j=0\ldots N-1} \delta^{t-1}(S_j)\,P(S_i|S_j)\,P(O^t|S_i) = P(O^t|S_i) \max_{j=0\ldots N-1} \delta^{t-1}(S_j)\,P(S_i|S_j)$
    
*  Déterminer quel est l'état au temps précédent qui a maximisé le score :  
     
    *   $\text{bp}^t(S_i) = \text{arg max}_{j=0\ldots N-1} \delta^{t-1}(S_j)\,P(S_i|S_j)$

*   Terminaison

    *   On obtient finalement la probabilité de la séquence complète : 

        *  $P^*(O) = \max_{j=0\ldots N-1} \delta^{T-1}(S_j)$
    
    *   Et l'état optimal : 
    
        *  $\text{bp}^{T-1}(S_i) = \text{arg max}_{j=0\ldots N-1} \delta^{T-2}(S_j)\,P(S_i|S_j)$


Puis pour récupérer le meilleur chemin d'états (*Backtracing*) :

   *   On choisit l'état appelé $S^{T-1}_*$ qui maximise $\delta^{T-1}$.
   *   Puis à l'instant $t$ précédent prendre : $S^{T-2} = \text{bp}^{T-1}(S^{T-1}_*)$.
   *   De manière générale, $S^{t-1} = \text{bp}^{t}(S^{t}_*)$.

   

#### Exemple POS tagging


<img src="https://www.irit.fr/~Thomas.Pellegrini/ens/M2ML2/cours2/viterbi_example_POS.png" alt="viterbi POs example" width="400"/>

(Image de Noah Smith)

#### Code **exercice**

Pour le backtracing, vous utiliserez la fonction ```torch.argmax()``` : 

https://pytorch.org/docs/stable/generated/torch.argmax.html

In [70]:
def viterbi_complet(observations, transition_matrix, emission_matrix, pi0):
    # observations : sequence d'observations, liste d'entiers
    # transition_matrix : matrice de transition N x N où N nb d'états cahcés
    # emission_matrix : matrice d'émission O x N où O nb d'observations possibles différentes
    # pi0: vecteur distribution stationnaire des N états cachés    
    num_states = transition_matrix.size(0) # nb d'états cachés
    num_observations = len(observations)  # longueur de la seq 
    
    alpha = torch.zeros((num_observations, num_states))
    # on ajoute ce tenseur pour stocker les meilleurs etats au long du scoring (backpointers)
    bp = torch.zeros((num_observations, num_states), dtype=torch.int)   
    # initialisation
    alpha[0] = pi0 * emission_matrix[observations[0]]
    # on ajoute ce tenseur pour stocker les meilleurs etats au long du scoring (backpointers)
    for t in range(1,num_observations):
        for i in range(num_states):
            prob = 0
            max_prob = 0
            max_state = 0
            for j in range(num_states):
                #print(f'alpha_{t-1}(x_{j}) * PTr({i})|{j}) * PEm({t}|{i})')       
                prob = alpha[t-1,j] * transition_matrix[i,j] * emission_matrix[observations[t],i]
                #print(f' {prob} = {alpha[t-1,j]} * {transition_matrix[i, j]} * {emission_matrix[observations[t],i]}')   
                if prob > max_prob:
                    max_prob = prob
                    max_state = j  
            #print(f'max_prob : {max_prob} max_state : {max_state}')           
            alpha[t,i] = max_prob
            bp[t, i] = max_state
    # terminaison scoring
    res = alpha[-1, torch.argmax(alpha[-1])].item()
    # backtracing
    viterbi_seq = [bp[0, torch.argmax(bp[0])].item()]
    for t in range(num_observations-1):
        current_state = torch.argmax(alpha[t-1])
        viterbi_seq.append(current_state.item())

    return res, viterbi_seq

In [72]:
#observations = [0,1,1,1,0,0]
observations = [0,1]
seq_obs_mots = [ind2obs[i] for i in observations]

score, seq_etats = viterbi_complet(observations, transition_matrix, emission_matrix, pi0)
print(" -> ".join(seq_obs_mots))
print("Score : {:.5f}".format(score))
seq_etats_mots = [ind2state[int(i)] for i in seq_etats]
print("Séquence d'états Viterbi : ", " -> ".join(seq_etats_mots))

if seq_etats == [0,2]:
  print('OK !')
else:
  print('KO !')


Canapé -> Tennis
Score : 0.05702
Séquence d'états Viterbi :  Pluie -> Soleil
OK !


<img src="/Users/asriel/TP_IAFA/M2_IAFA/TP_M2IAFA/TP_M2_APPAUT/TP1/viterbi_inference.png" alt="viterbi l'inférence ou le décodage" width="600"/>

:::

Calcul explicite jusqu’à 0,00118522
On suit le chemin Viterbi A→B→B→End pour les observations x,y,y.​​

 - $π_A$ est la probabilité initiale de commencer en A
 - $a_{u,v}=P(état_{t}=v∣état_{t-1}=u$ les transitions,
 - $b_{s}(o)=P(obs=o∣état=s)$ les émissions,
 - $a_{B,End}$ a transition finale vers End


Les égalités données sont :     
 - $δ_1(A)=π_A × b_A(x)=0,7$
 - $δ_2(B)=δ_1(A) × a_{A,B} × b_B(y) = 0,196$
 - $δ_3(B)=δ_2(B)× a_{B,B} × b_B(y) = 0,02646$
 - $δ_4=δ_3(B) × a_{B,End} = 0,00118522$​​

On peut donc retrouver les paramètres implicites :

 - À l’étape 2 : $0,196 = 0,7 × aA,B × bB(y) ⇒ aA,B x bB(y) = 0,196/0,7 ≈ 0,28$.
 - À l’étape 3 : $0,02646 = 0,196 × aB,B × bB(y)⇒ aB,B x bB(y) = 0,02646/0,196 ≈ 0,135$.
 - À l’étape 4 : $0,00118522/0,02646 ≈ 0,0448 ⇒ aB,End = 0,00118522/0,02646 ≈ 0,0448$.

Donc, numériquement, le calcul complet sur le chemin Viterbi est
 -  $π_A × b_A(x) × a_{A,B} × b_B(y) × a_{B,B} × b_B(y) × a_{B,End} = 0,7×0,28×0,135×0,0448≈0,00118522$ 
 
:::